# MovieLens 25M full training run

Run the cells from top to bottom in a Google Colab runtime. Colab uses `/content` paths; place the six CSV files in `MyDrive/recsys/data/raw/`. This workflow loads the full dataset into SQLite, trains all classical models, and keeps MLflow and exported results on Drive.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/sxxxsewan/MovieRecSys.git"
REPO_DIR = Path("/content/MovieRecSys")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive" / "recsys"
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / "data" / "raw"
DATA_DIR = REPO_DIR / "data"
OUTPUT_DIR = DRIVE_PROJECT_DIR / "outputs"
MLFLOW_DIR = DRIVE_PROJECT_DIR / "mlruns"
DATABASE_PATH = DATA_DIR / "recsys.sqlite"
REGISTRY_PATH = DRIVE_PROJECT_DIR / "mlruns_registry.db"
CSV_FILES = (
    "ratings.csv",
    "movies.csv",
    "tags.csv",
    "links.csv",
    "genome-scores.csv",
    "genome-tags.csv",
)
print(f"Repository: {REPO_DIR}")
print(f"Drive CSV source: {DRIVE_DATA_DIR}")
print(f"SQLite database: {DATABASE_PATH}")
print(f"Drive outputs: {OUTPUT_DIR}")

## 1. Setup

Clone or update the repository and install its pinned project and development dependencies.

In [ ]:
import subprocess
import sys

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[dev]"],
    cwd=REPO_DIR,
    check=True,
)

## 2. Data

Mount Drive, validate every required MovieLens CSV, and copy the files to the repository's configured data directory.

In [ ]:
import shutil

from google.colab import drive

drive.mount(str(DRIVE_MOUNT))
missing_files = [name for name in CSV_FILES if not (DRIVE_DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Missing MovieLens CSV files in {DRIVE_DATA_DIR}: {', '.join(missing_files)}. "
        "Upload the six files to MyDrive/recsys/data/raw/ and rerun this cell."
    )
DATA_DIR.mkdir(parents=True, exist_ok=True)
for filename in CSV_FILES:
    shutil.copy2(DRIVE_DATA_DIR / filename, DATA_DIR / filename)
print(f"Copied {len(CSV_FILES)} CSV files into {DATA_DIR}")

## 3. Configuration and full run

Use SQLite for the application database and Drive-backed file stores for MLflow tracking, registry, and model artifacts. Apply migrations, load the full CSV dataset, then train and evaluate all models.

In [ ]:
import os
import subprocess
import sys

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["MOVIELENS_DATA_DIR"] = str(DATA_DIR)
os.environ["DATABASE_URL"] = f"sqlite:///{DATABASE_PATH}"
os.environ["MLFLOW_TRACKING_URI"] = f"file:{MLFLOW_DIR}"
os.environ["MLFLOW_REGISTRY_URI"] = f"sqlite:///{REGISTRY_PATH}"

subprocess.run(["alembic", "upgrade", "head"], cwd=REPO_DIR, check=True)
subprocess.run(
    [
        sys.executable,
        "-m",
        "recsys.data.loader",
        "--config",
        "configs/data.yaml",
        "--replace",
    ],
    cwd=REPO_DIR,
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "recsys.training",
        "--data-config",
        "configs/data.yaml",
        "--model-config",
        "configs/models.yaml",
        "--experiment-config",
        "configs/experiments.yaml",
    ],
    cwd=REPO_DIR,
    check=True,
)

## 4. Results and saved artifacts

Show the latest MLflow run for each model, save validation and test metrics to Drive, export the validation-selected model, and save an interactive comparison plot.

In [ ]:
import mlflow
import pandas as pd
import plotly.express as px
from mlflow import MlflowClient

experiment_name = "movielens-25m"
client = MlflowClient(tracking_uri=os.environ["MLFLOW_TRACKING_URI"])
experiment = client.get_experiment_by_name(experiment_name)
if experiment is None:
    raise RuntimeError(f"MLflow experiment not found: {experiment_name}")
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"],
    max_results=100,
)
latest_runs = {}
for run in runs:
    latest_runs.setdefault(run.info.run_name or run.info.run_id, run)
metrics_frame = pd.DataFrame(
    [
        {"model": name, "run_id": run.info.run_id, **run.data.metrics}
        for name, run in latest_runs.items()
    ]
)
if metrics_frame.empty:
    raise RuntimeError("No completed model runs were found in MLflow")
metrics_frame = metrics_frame.sort_values(
    "validation_ndcg_at_10", ascending=False
).reset_index(drop=True)
metrics_path = OUTPUT_DIR / "model_metrics.csv"
metrics_frame.to_csv(metrics_path, index=False)
display(metrics_frame)

best_run = max(
    latest_runs.values(),
    key=lambda run: run.data.metrics.get("validation_ndcg_at_10", float("-inf")),
)
best_model_dir = OUTPUT_DIR / "best_model"
downloaded_model = mlflow.artifacts.download_artifacts(
    run_id=best_run.info.run_id,
    artifact_path="model",
    dst_path=str(best_model_dir),
)
plot_path = OUTPUT_DIR / "model_metrics.html"
figure = px.bar(
    metrics_frame,
    x="model",
    y="test_ndcg_at_10",
    color="validation_ndcg_at_10",
    title="MovieLens 25M model comparison: NDCG@10",
)
figure.write_html(plot_path)
display(figure)
print(f"Metrics CSV: {metrics_path}")
print(f"Interactive plot: {plot_path}")
print(f"Best model ({best_run.info.run_name}) exported to: {downloaded_model}")
print(f"MLflow run artifacts remain under: {MLFLOW_DIR}")